# MedScope 3D — multi-GPU triage training (Kaggle, 2× GPU)

Trains the urgency-triage XGBoost model across **both** Kaggle GPUs (e.g. T4 ×2 or P100 ×2) using a **Dask-CUDA** cluster with one worker per GPU and `xgboost.dask`.

> ⚠️ **Research/education only.** The dataset is *synthetic* and rule-seeded — accuracy here is **not** clinical accuracy. Emergency recall is a hard release gate; in production the deterministic rule engine also backstops the model (defense in depth). See `deploy/docs/MODEL_CARD.md`.

**Setup:** Notebook settings → Accelerator → **GPU T4 ×2** (or P100 ×2). Make the repo available either by (a) cloning your GitHub fork, or (b) adding it as a Kaggle Dataset and pointing `REPO_DIR` at it.

In [ ]:
# 1. Make the repo importable. Option A: git clone your fork. Option B: Kaggle Dataset.
import os, subprocess, sys
REPO_URL = os.environ.get('MEDSCOPE_REPO_URL', '')  # e.g. https://github.com/<you>/medscope-3d
REPO_DIR = '/kaggle/working/medscope-3d'
if REPO_URL and not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
# If you added the repo as a Kaggle Dataset instead, set REPO_DIR to that path, e.g.:
# REPO_DIR = '/kaggle/input/medscope-3d'
print('REPO_DIR =', REPO_DIR)
assert os.path.isdir(REPO_DIR), 'Set REPO_URL or point REPO_DIR at the repo (Kaggle Dataset).'

In [ ]:
# 2. Install the shared contracts + ML package and the multi-GPU stack.
!pip -q install -e {REPO_DIR}/packages/triage-shared/python -e {REPO_DIR}/ml
!pip -q install 'dask-cuda>=24.2' 'dask[complete]>=2024.1'
import xgboost as xgb; print('xgboost', xgb.__version__)

In [ ]:
# 3. Confirm both GPUs are visible.
!nvidia-smi --query-gpu=index,name,memory.total --format=csv,noheader

In [ ]:
# 4. Generate a LARGE synthetic, safety-biased dataset (CPU) and build features.
import numpy as np, yaml
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from medscope_ml import features as F
from medscope_ml.generate import generate_dataframe

cfg = yaml.safe_load(Path(REPO_DIR, 'ml', 'config.yaml').read_text())
N_ROWS = 300_000       # scale up for the GPUs
SEED = cfg['dataset']['seed']
df = generate_dataframe(N_ROWS, SEED, cfg['dataset']['redflag_target_fraction'])
cols = F.feature_columns()
X = df[cols].to_numpy(dtype=np.float32)
le = LabelEncoder(); y = le.fit_transform(df['urgency'].to_numpy()); classes = list(le.classes_)
print('rows', len(df), 'classes', classes)
print('class distribution:', df['urgency'].value_counts().to_dict())

In [ ]:
# 5. Splits + EMERGENCY-upweighted sample weights (safety).
ds = cfg['dataset']
X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=ds['test_size']+ds['val_size'], random_state=SEED, stratify=y)
rel_val = ds['val_size']/(ds['test_size']+ds['val_size'])
X_val, X_te, y_val, y_te = train_test_split(X_tmp, y_tmp, test_size=1-rel_val, random_state=SEED, stratify=y_tmp)
ei = classes.index('EMERGENCY')
w = np.ones(len(y_tr), np.float32)
w[y_tr == ei] = cfg['train'].get('emergency_weight', 5.0)
w[y_tr == classes.index('URGENT_TODAY')] = cfg['train'].get('urgent_weight', 1.5)

In [ ]:
# 6. Spin up a Dask-CUDA cluster: ONE worker PER GPU -> uses ALL visible GPUs.
from dask_cuda import LocalCUDACluster
from dask.distributed import Client
import dask.array as da

cluster = LocalCUDACluster()            # auto-detects every GPU (2 on Kaggle T4x2)
client = Client(cluster)
n_gpus = len(cluster.cuda_visible_devices) if hasattr(cluster, 'cuda_visible_devices') else len(client.scheduler_info()['workers'])
print('Dask-CUDA workers (GPUs):', len(client.scheduler_info()['workers']))
client

In [ ]:
# 7. Train XGBoost across BOTH GPUs with xgboost.dask.
import xgboost as xgb
chunks = (max(1, len(X_tr)//(2*4)), -1)   # chunk so work spreads across the 2 GPU workers
dX = da.from_array(X_tr, chunks=chunks)
dy = da.from_array(y_tr, chunks=chunks[0])
dw = da.from_array(w,    chunks=chunks[0])
dtrain = xgb.dask.DaskDMatrix(client, dX, dy, weight=dw)
xcfg = cfg['train']['xgboost']
params = {'objective':'multi:softprob','num_class':len(classes),'tree_method':'hist','device':'cuda',
          'max_depth':xcfg['max_depth'],'eta':xcfg['learning_rate'],'subsample':xcfg['subsample'],
          'colsample_bytree':xcfg['colsample_bytree'],'eval_metric':'mlogloss'}
output = xgb.dask.train(client, params, dtrain, num_boost_round=xcfg['n_estimators'])
booster = output['booster']
print('trained; best rounds:', xcfg['n_estimators'])

In [ ]:
# 8. Tune the EMERGENCY decision threshold on validation, then evaluate the gate on test.
from medscope_ml import safety_eval as S, metrics as M
dval = xgb.dask.DaskDMatrix(client, da.from_array(X_val, chunks=chunks), da.from_array(y_val, chunks=chunks[0]))
proba_val = xgb.dask.predict(client, output, dval).compute()
target = cfg['evaluation']['emergency_recall_min']
true_em = proba_val[y_val==ei, ei]
tau_e = float(min(max(np.quantile(true_em, max(0.0, 1-target-0.01)), 0.05), 0.5)) if len(true_em) else 0.5

dtest = xgb.dask.DaskDMatrix(client, da.from_array(X_te, chunks=chunks), da.from_array(y_te, chunks=chunks[0]))
proba_te = xgb.dask.predict(client, output, dtest).compute()
pred = proba_te.argmax(1); pred[proba_te[:,ei] >= tau_e] = ei
y_true = [classes[i] for i in y_te]; y_pred = [classes[i] for i in pred]
std = M.compute_metrics(y_true, y_pred, proba_te, classes)
safety = S.safety_report(y_true, y_pred, target)
print('tau_e', round(tau_e,3), '| macro_f1', round(std['macro_f1'],4))
print('EMERGENCY recall', round(safety['emergency_recall'],4), 'gate', 'PASS' if safety['gate_passed'] else 'FAIL')
assert safety['gate_passed'], 'Emergency-recall gate failed — do not ship this artifact.'

In [ ]:
# 9. Save an artifact compatible with medscope_ml.predict / the Phase 3 PredictionService.
import joblib, json, datetime

class BoosterSklearnAdapter:
    """Wrap a raw Booster to expose predict_proba(X)->(n, n_classes) for serving."""
    def __init__(self, booster, n_classes):
        self.booster = booster; self.n_classes = n_classes
    def predict_proba(self, X):
        import xgboost as xgb, numpy as np
        return self.booster.inplace_predict(np.ascontiguousarray(X, dtype=np.float32))

artifact = {
    'model': BoosterSklearnAdapter(booster, len(classes)),
    'feature_columns': cols, 'classes': classes, 'algo': 'xgboost-dask-gpu',
    'model_version': cfg['artifacts']['model_version'],
    'confidence_threshold': cfg['evaluation']['confidence_threshold'],
    'emergency_threshold': tau_e, 'device_trained': 'cuda-multi',
    'trained_at': datetime.datetime.utcnow().isoformat(),
    'dataset': {'n_rows': int(N_ROWS), 'seed': SEED}, 'metrics': std, 'safety': safety,
}
out = f"/kaggle/working/model_{cfg['artifacts']['model_version']}.joblib"
joblib.dump(artifact, out)
json.dump({'metrics': std, 'safety': safety, 'tau_e': tau_e}, open('/kaggle/working/metrics.json','w'), indent=2)
print('saved', out)
client.close(); cluster.close()